# B2.6 · Sub-agents and delegation depth

**Function B — Product & Application Security → The Security Automation / Harness Engineer**  ·  *Security of AI*

---

**Risk.** Authority inherited silently from parent to child agent.

**Control.** Fan-out control, recursion budgets, explicit authority inheritance.

**This lab.** Prove a grandchild agent cannot exceed its parent.

| | |
|---|---|
| Open-source tooling | kagent, SPIRE |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("B2.6"))

Sub-agents multiply capability and delegation depth at the same time. Only one of those is on the roadmap.

In [ ]:
from cybercommons import identity

reg = identity.Registry()
root = reg.record(identity.mint("alice"))
orch = reg.record(identity.exchange(root, "patch-agent", {"repo:read", "repo:write"}))
sub  = reg.record(identity.exchange(orch, "deploy-agent", {"repo:read"}))

for t in (root, orch, sub):
    print(f"depth {len(t.chain())}  {' → '.join(t.chain()):46s} {sorted(t.scopes)}")

MAX_DEPTH = 3
deepest = max(len(t.chain()) for t in reg.issued)
print(f"\ndeepest chain {deepest} (limit {MAX_DEPTH}) — "
      f"{'ok' if deepest <= MAX_DEPTH else 'REFUSE further delegation'}")

Depth is easy to bound and almost never bounded. The reason it matters: each hop is a place where a widening bug would apply, and the deepest chain is the one nobody drew on the architecture diagram.

In [ ]:
# a sub-agent cannot exceed what its parent presented
try:
    identity.exchange(sub, "deploy-agent", {"repo:write"})
except identity.DelegationError as e:
    print("sub-agent tried to widen:", e)

### Expect

Three tokens print at depths 1–3 with narrowing scopes, the depth check passes at the limit, and the sub-agent's attempt to regain `repo:write` is refused.

### Your turn

Where should the depth limit be enforced — the orchestrator, the token issuer, or the resource server? Only one of those still works when the orchestrator is the compromised component.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/B2.6.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*